In [30]:
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env file from project root

BLS_API_KEY    = os.environ.get('BLS_API_KEY')
CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY')

In [41]:
df = pd.read_csv('data/raw/Zip_zori_uc_sfrcondomfr_sm_month.csv')

In [42]:
print(df.columns.tolist())

['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName', '2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30', '2015-05-31', '2015-06-30', '2015-07-31', '2015-08-31', '2015-09-30', '2015-10-31', '2015-11-30', '2015-12-31', '2016-01-31', '2016-02-29', '2016-03-31', '2016-04-30', '2016-05-31', '2016-06-30', '2016-07-31', '2016-08-31', '2016-09-30', '2016-10-31', '2016-11-30', '2016-12-31', '2017-01-31', '2017-02-28', '2017-03-31', '2017-04-30', '2017-05-31', '2017-06-30', '2017-07-31', '2017-08-31', '2017-09-30', '2017-10-31', '2017-11-30', '2017-12-31', '2018-01-31', '2018-02-28', '2018-03-31', '2018-04-30', '2018-05-31', '2018-06-30', '2018-07-31', '2018-08-31', '2018-09-30', '2018-10-31', '2018-11-30', '2018-12-31', '2019-01-31', '2019-02-28', '2019-03-31', '2019-04-30', '2019-05-31', '2019-06-30', '2019-07-31', '2019-08-31', '2019-09-30', '2019-10-31', '2019-11-30', '2019-12-31', '2020-01-31', '2020-02-29', '2020-03-31', '2020-04-30'

In [43]:
df.shape

(7782, 142)

In [44]:
# Filter for Los Angeles metro only
la_df = df[df['Metro'] == 'Los Angeles-Long Beach-Anaheim, CA']

# See what we have
print(f"Total LA ZIP codes: {len(la_df)}")
print(la_df[['RegionName', 'City', 'CountyName']].to_string())

Total LA ZIP codes: 325
      RegionName                    City          CountyName
9          90011             Los Angeles  Los Angeles County
12         90650                 Norwalk  Los Angeles County
13         91331             Los Angeles  Los Angeles County
21         90044             Los Angeles  Los Angeles County
26         90201                    Bell  Los Angeles County
27         90250               Hawthorne  Los Angeles County
30         90805              Long Beach  Los Angeles County
34         90280              South Gate  Los Angeles County
36         91342             Los Angeles  Los Angeles County
37         92683             Westminster       Orange County
66         92804                 Anaheim       Orange County
93         91744               La Puente  Los Angeles County
100        93550                Palmdale  Los Angeles County
111        90706              Bellflower  Los Angeles County
114        93535               Lancaster  Los Angeles County


In [45]:
# Filter for City == Los Angeles only to narrow further
la_city_df = la_df[la_df['City'] == 'Los Angeles']

print(f"ZIP codes in City of Los Angeles: {len(la_city_df)}")
print(la_city_df[['RegionName', 'City', 'CountyName']].to_string())

ZIP codes in City of Los Angeles: 95
      RegionName         City          CountyName
9          90011  Los Angeles  Los Angeles County
13         91331  Los Angeles  Los Angeles County
21         90044  Los Angeles  Los Angeles County
36         91342  Los Angeles  Los Angeles County
124        91335  Los Angeles  Los Angeles County
158        90003  Los Angeles  Los Angeles County
242        90026  Los Angeles  Los Angeles County
251        91402  Los Angeles  Los Angeles County
283        90037  Los Angeles  Los Angeles County
338        90731  Los Angeles  Los Angeles County
348        91343  Los Angeles  Los Angeles County
418        90019  Los Angeles  Los Angeles County
469        90004  Los Angeles  Los Angeles County
493        90042  Los Angeles  Los Angeles County
563        90006  Los Angeles  Los Angeles County
586        90066  Los Angeles  Los Angeles County
624        90744  Los Angeles  Los Angeles County
636        91406  Los Angeles  Los Angeles County
658        91

In [46]:
# Neighborhood to ZIP code mapping
neighborhood_zip_map = {
    'University Park':      ['90007'],
    'Exposition Park':      ['90037'],
    'Jefferson Park':       ['90018'],
    'Vermont Square':       ['90044'],
    'Pico-Union':           ['90006'],
    'Koreatown':            ['90005', '90006', '90010'],
    'West Adams':           ['90016', '90018'],
    'Leimert Park':         ['90008'],
    'Boyle Heights':        ['90033', '90023'],
    'Echo Park':            ['90026'],
    'Silver Lake':          ['90026', '90039'],
    'Los Feliz':            ['90027'],
    'Highland Park':        ['90042'],
    'Glassell Park':        ['90065'],
    'Culver City':          ['90230', '90232'],
    'Mar Vista':            ['90066'],
    'Palms':                ['90034'],
    'West Hollywood':       ['90046', '90069'],
    'Mid-Wilshire':         ['90036'],
    'Hancock Park':         ['90004'],
    'Beverly Hills':        ['90210', '90211'],
    'Brentwood':            ['90049'],
    'Santa Monica':         ['90401', '90403', '90405'],
    'Westwood':             ['90024'],
    'Downtown LA':          ['90012', '90014', '90017'],
}

# Get all unique ZIP codes we need
all_zips = [zip for zips in neighborhood_zip_map.values() for zip in zips]
all_zips = list(set(all_zips))

print(f"Total unique ZIP codes needed: {len(all_zips)}")
print(sorted(all_zips))

Total unique ZIP codes needed: 34
['90004', '90005', '90006', '90007', '90008', '90010', '90012', '90014', '90016', '90017', '90018', '90023', '90024', '90026', '90027', '90033', '90034', '90036', '90037', '90039', '90042', '90044', '90046', '90049', '90065', '90066', '90069', '90210', '90211', '90230', '90232', '90401', '90403', '90405']


In [47]:
# Filter Zillow data to only our neighborhood ZIP codes
# RegionName in Zillow is stored as integer, so convert our zips
all_zips_int = [int(z) for z in all_zips]

# Filter
neighborhood_df = la_df[la_df['RegionName'].isin(all_zips_int)]

print(f"ZIP codes found in Zillow data: {len(neighborhood_df)}")
print(f"ZIP codes missing from Zillow data: {len(all_zips) - len(neighborhood_df)}")

# Show which ones we have
found_zips = neighborhood_df['RegionName'].astype(str).tolist()
missing_zips = [z for z in all_zips if z not in found_zips]

print(f"\nFound ZIPs: {sorted(found_zips)}")
print(f"\nMissing ZIPs: {sorted(missing_zips)}")

ZIP codes found in Zillow data: 34
ZIP codes missing from Zillow data: 0

Found ZIPs: ['90004', '90005', '90006', '90007', '90008', '90010', '90012', '90014', '90016', '90017', '90018', '90023', '90024', '90026', '90027', '90033', '90034', '90036', '90037', '90039', '90042', '90044', '90046', '90049', '90065', '90066', '90069', '90210', '90211', '90230', '90232', '90401', '90403', '90405']

Missing ZIPs: []


In [ ]:
# Get only the date columns (they start with '20')
date_cols = [col for col in df.columns if col.startswith('20')]

# Filter to only 2019 onwards
date_cols = [col for col in date_cols if col >= '2019-01-01']

# Keep only relevant columns
keep_cols = ['RegionName'] + date_cols
neighborhood_df = neighborhood_df[keep_cols].copy()

# Convert RegionName to string for mapping
neighborhood_df['RegionName'] = neighborhood_df['RegionName'].astype(str).str.zfill(5)

# Reverse the mapping: ZIP -> Neighborhood
zip_to_neighborhood = {}
for neighborhood, zips in neighborhood_zip_map.items():
    for z in zips:
        zip_to_neighborhood[z] = neighborhood

# Add neighborhood column
neighborhood_df['Neighborhood'] = neighborhood_df['RegionName'].map(zip_to_neighborhood)

# Backward fill missing rent values per ZIP code before averaging
neighborhood_df[date_cols] = neighborhood_df[date_cols].bfill(axis=1)

# Group by neighborhood and average across ZIP codes
neighborhood_avg = neighborhood_df.groupby('Neighborhood')[date_cols].mean()

print(f"Shape: {neighborhood_avg.shape}")
print(f"Neighborhoods: {neighborhood_avg.index.tolist()}")
print(neighborhood_avg.iloc[:, :3])

In [49]:
# Check how many NaN months each neighborhood has
nan_counts = neighborhood_avg.isna().sum(axis=1)
print("NaN months per neighborhood:")
print(nan_counts[nan_counts > 0].sort_values(ascending=False))

# Check when data actually starts for the problematic neighborhoods
problem_neighborhoods = nan_counts[nan_counts > 0].index.tolist()
for n in problem_neighborhoods:
    first_valid = neighborhood_avg.loc[n].first_valid_index()
    print(f"{n}: data starts at {first_valid}")

NaN months per neighborhood:
Neighborhood
Exposition Park    41
University Park    36
Boyle Heights      24
Vermont Square     22
Glassell Park       9
Culver City         1
dtype: int64
Boyle Heights: data starts at 2021-01-31
Culver City: data starts at 2019-02-28
Exposition Park: data starts at 2022-06-30
Glassell Park: data starts at 2019-10-31
University Park: data starts at 2022-01-31
Vermont Square: data starts at 2020-11-30


In [52]:
# Save the dataset to CSV
neighborhood_avg.to_csv('data/processed/la_neighborhood_rental_prices.csv')
print("Dataset saved to data/processed/la_neighborhood_rental_prices.csv")

# Also save with neighborhood as a column instead of index for easier viewing
neighborhood_avg_reset = neighborhood_avg.reset_index()
neighborhood_avg_reset.to_csv('data/processed/la_neighborhood_rental_prices_with_index.csv', index=False)
print("Dataset with index saved to data/processed/la_neighborhood_rental_prices_with_index.csv")

Dataset saved to data/processed/la_neighborhood_rental_prices.csv
Dataset with index saved to data/processed/la_neighborhood_rental_prices_with_index.csv


In [59]:
import requests
import pandas as pd
from io import StringIO

# Public permits dataset - 2020 to present, no auth required
url = "https://data.lacity.org/resource/pi9x-tg5x.csv"
params = {
    '$limit': 50000,
    '$order': 'issue_date DESC'
}

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    permits_df = pd.read_csv(StringIO(response.text))
    print(f"Shape: {permits_df.shape}")
    print(f"Columns: {permits_df.columns.tolist()}")
    print(permits_df.head(3))
else:
    print(f"Error: {response.text[:300]}")

Status code: 200
Shape: (50000, 38)
Columns: ['permit_nbr', 'primary_address', 'zip_code', 'cd', 'pin_nbr', 'apn', 'zone', 'apc', 'cpa', 'cnc', 'hl', 'ct', 'permit_group', 'permit_type', 'permit_sub_type', 'use_code', 'use_desc', 'submitted_date', 'issue_date', 'cofo_date', 'du_changed', 'adu_changed', 'junior_adu', 'square_footage', 'status_desc', 'status_date', 'valuation', 'construction', 'height', 'type_lat_lon', 'lat', 'lon', 'work_desc', 'ev', 'solar', 'business_unit', 'refresh_time', 'geolocation']
          permit_nbr       primary_address  zip_code    cd        pin_nbr  \
0  26016-90000-06207        1359 W 27TH PL   90731.0  15.0  009B193   692   
1  26016-90000-06206     3963 N EVADALE DR   90031.0   1.0  147A227    34   
2  25016-10001-19741  5057 N NEWCASTLE AVE   91316.0   4.0  171B125   845   

          apn          zone               apc                    cpa  \
0  7470010022        R1-1XL            Harbor              San Pedro   
1  5303009031  [Q]R1-1D-HCR  East Lo

/var/folders/hk/svxlhg9x51j4t3tpg3zjg__00000gn/T/ipykernel_18937/973713094.py:16: DtypeWarning: Columns (0: ct) have mixed types. Specify dtype option on import or set low_memory=False.
  permits_df = pd.read_csv(StringIO(response.text))


In [60]:
permits_df.columns.to_list()

['permit_nbr',
 'primary_address',
 'zip_code',
 'cd',
 'pin_nbr',
 'apn',
 'zone',
 'apc',
 'cpa',
 'cnc',
 'hl',
 'ct',
 'permit_group',
 'permit_type',
 'permit_sub_type',
 'use_code',
 'use_desc',
 'submitted_date',
 'issue_date',
 'cofo_date',
 'du_changed',
 'adu_changed',
 'junior_adu',
 'square_footage',
 'status_desc',
 'status_date',
 'valuation',
 'construction',
 'height',
 'type_lat_lon',
 'lat',
 'lon',
 'work_desc',
 'ev',
 'solar',
 'business_unit',
 'refresh_time',
 'geolocation']

In [61]:
# Keep only relevant columns
cols_to_keep = [
    'permit_nbr', 'zip_code', 'permit_type', 'permit_sub_type',
    'use_desc', 'issue_date', 'du_changed', 'adu_changed',
    'junior_adu', 'square_footage', 'valuation', 'lat', 'lon'
]

permits_clean = permits_df[cols_to_keep].copy()

# See what permit types exist
print("Permit types:")
print(permits_clean['permit_type'].value_counts())

print("\nPermit sub types:")
print(permits_clean['permit_sub_type'].value_counts().head(20))

print("\nUse descriptions:")
print(permits_clean['use_desc'].value_counts().head(20))

Permit types:
permit_type
Bldg-Alter/Repair       33121
Bldg-Addition            4102
Bldg-New                 3767
Grading                  2790
Swimming-Pool/Spa        1726
Nonbldg-New              1573
Bldg-Demolition          1370
Sign                     1068
Nonbldg-Alter/Repair      448
Nonbldg-Addition           19
Nonbldg-Demolition         13
Bldg-Relocation             3
Name: count, dtype: int64

Permit sub types:
permit_sub_type
1 or 2 Family Dwelling    37741
Apartment                  6069
Commercial                 5122
Onsite                      936
Offsite                     132
Name: count, dtype: int64

Use descriptions:
use_desc
Dwelling - Single Family        23058
Accessory Dwelling Unit          5019
Apartment                        4442
Grading - Hillside               2044
Duplex                           1985
Pool/Spa - Private               1795
Garage - Private                 1057
Office                            986
Retaining Wall                    8

In [23]:
import requests
import pandas as pd
from io import StringIO

url = "https://data.lacity.org/resource/pi9x-tg5x.csv"
params = {
    '$limit': 50000,
    '$order': 'issue_date DESC'
}

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    permits_df = pd.read_csv(StringIO(response.text))
    print(f"Shape: {permits_df.shape}")
    print(permits_df['permit_type'].value_counts())
    print(permits_df['use_desc'].value_counts().head(20))
else:
    print(f"Error: {response.text[:300]}")

Status code: 200
Shape: (50000, 38)
permit_type
Bldg-Alter/Repair       33158
Bldg-Addition            4019
Bldg-New                 3785
Grading                  2837
Swimming-Pool/Spa        1735
Nonbldg-New              1550
Bldg-Demolition          1383
Sign                     1066
Nonbldg-Alter/Repair      432
Nonbldg-Addition           21
Nonbldg-Demolition         12
Bldg-Relocation             2
Name: count, dtype: int64
use_desc
Dwelling - Single Family        22953
Accessory Dwelling Unit          5034
Apartment                        4418
Grading - Hillside               2086
Duplex                           1992
Pool/Spa - Private               1801
Garage - Private                 1035
Office                            979
Retaining Wall                    816
Grading - Non-Hillside            749
Demolition                        717
Miscellaneous Bldg/Structure      706
Wall Sign                         659
Condo-Multi Family                572
Retail                   

/var/folders/hk/svxlhg9x51j4t3tpg3zjg__00000gn/T/ipykernel_95040/3724805132.py:15: DtypeWarning: Columns (0: ct) have mixed types. Specify dtype option on import or set low_memory=False.
  permits_df = pd.read_csv(StringIO(response.text))


In [27]:
permits_df.columns.to_list()

['permit_nbr',
 'primary_address',
 'zip_code',
 'cd',
 'pin_nbr',
 'apn',
 'zone',
 'apc',
 'cpa',
 'cnc',
 'hl',
 'ct',
 'permit_group',
 'permit_type',
 'permit_sub_type',
 'use_code',
 'use_desc',
 'submitted_date',
 'issue_date',
 'cofo_date',
 'du_changed',
 'adu_changed',
 'junior_adu',
 'square_footage',
 'status_desc',
 'status_date',
 'valuation',
 'construction',
 'height',
 'type_lat_lon',
 'lat',
 'lon',
 'work_desc',
 'ev',
 'solar',
 'business_unit',
 'refresh_time',
 'geolocation']

In [26]:
# Filter to residential permit types only
residential_types = ['Bldg-New', 'Bldg-Addition', 'Bldg-Alter/Repair', 'Bldg-Demolition']
residential_uses = [
    'Dwelling - Single Family',
    'Accessory Dwelling Unit',
    'Apartment',
    'Duplex',
    'Condominium'
]

permits_filtered = permits_df[
    (permits_df['permit_type'].isin(residential_types)) &
    (permits_df['use_desc'].isin(residential_uses))
].copy()

# Keep only columns we need
cols_to_keep = [
    'permit_nbr', 'zip_code', 'permit_type', 'use_desc',
    'issue_date', 'du_changed', 'adu_changed', 'junior_adu',
    'valuation', 'lat', 'lon'
]
permits_filtered = permits_filtered[cols_to_keep]

# Convert issue_date to datetime
permits_filtered['issue_date'] = pd.to_datetime(permits_filtered['issue_date'])

print(f"Shape after filtering: {permits_filtered.shape}")
print(f"\nPermit types remaining:")
print(permits_filtered['permit_type'].value_counts())
print(f"\nDate range: {permits_filtered['issue_date'].min()} to {permits_filtered['issue_date'].max()}")
print(permits_filtered.head())

Shape after filtering: (34608, 11)

Permit types remaining:
permit_type
Bldg-Alter/Repair    27335
Bldg-Addition         3803
Bldg-New              3171
Bldg-Demolition        299
Name: count, dtype: int64

Date range: 2025-06-26 00:00:00 to 2026-04-12 00:00:00
          permit_nbr  zip_code        permit_type                  use_desc  \
0  26016-90000-09581   91436.0  Bldg-Alter/Repair  Dwelling - Single Family   
1  26016-90000-09599   90045.0  Bldg-Alter/Repair  Dwelling - Single Family   
2  26016-90000-09582   91311.0  Bldg-Alter/Repair               Condominium   
3  25016-10000-29224   90019.0  Bldg-Alter/Repair   Accessory Dwelling Unit   
4  26016-90000-09603   91423.0  Bldg-Alter/Repair  Dwelling - Single Family   

  issue_date  du_changed  adu_changed  junior_adu  valuation        lat  \
0 2026-04-12         NaN          NaN         NaN     4200.0  34.157790   
1 2026-04-12         NaN          NaN         NaN     9600.0  33.967040   
2 2026-04-12         NaN          NaN 

In [32]:
import time
from io import StringIO

url = "https://data.lacity.org/resource/pi9x-tg5x.csv"
all_permits = []

# Pull year by year - faster and more reliable
years = ['2020', '2021', '2022', '2023', '2024', '2025']

for year in years:
    params = {
        '$limit': 50000,
        '$where': f"issue_date >= '{year}-01-01' AND issue_date < '{int(year)+1}-01-01'",
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        batch = pd.read_csv(StringIO(response.text))
        all_permits.append(batch)
        print(f"{year}: {len(batch)} permits pulled")
    else:
        print(f"{year}: Error — {response.text[:100]}")
    
    time.sleep(1)

permits_full = pd.concat(all_permits, ignore_index=True)
print(f"\nTotal rows: {len(permits_full)}")
print(f"Date range: {pd.to_datetime(permits_full['issue_date']).min()} to {pd.to_datetime(permits_full['issue_date']).max()}")

2020: 50000 permits pulled
2021: 50000 permits pulled


/var/folders/hk/svxlhg9x51j4t3tpg3zjg__00000gn/T/ipykernel_95040/3054726823.py:19: DtypeWarning: Columns (0: ct) have mixed types. Specify dtype option on import or set low_memory=False.
  batch = pd.read_csv(StringIO(response.text))


2022: 50000 permits pulled
2023: 50000 permits pulled
2024: 50000 permits pulled
2025: 50000 permits pulled

Total rows: 300000
Date range: 2020-01-01 00:00:00 to 2025-10-11 00:00:00


In [33]:
# Check if any year hit exactly 50000 (means we may have missed records)
for year in years:
    year_count = len(permits_full[
        (pd.to_datetime(permits_full['issue_date']).dt.year == int(year))
    ])
    print(f"{year}: {year_count} permits")

2020: 50000 permits
2021: 50000 permits
2022: 50000 permits
2023: 50000 permits
2024: 50000 permits
2025: 50000 permits


In [34]:
import time
from io import StringIO

url = "https://data.lacity.org/resource/pi9x-tg5x.csv"
all_permits = []

years = ['2020', '2021', '2022', '2023', '2024', '2025']

for year in years:
    offset = 0
    year_total = 0
    
    while True:
        params = {
            '$limit': 50000,
            '$offset': offset,
            '$where': f"issue_date >= '{year}-01-01' AND issue_date < '{int(year)+1}-01-01'",
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"{year} offset {offset}: Error")
            break
            
        batch = pd.read_csv(StringIO(response.text))
        
        if len(batch) == 0:
            print(f"{year}: done — {year_total} total permits")
            break
            
        all_permits.append(batch)
        year_total += len(batch)
        offset += 50000
        print(f"{year}: pulled {year_total} so far...")
        time.sleep(1)

permits_full = pd.concat(all_permits, ignore_index=True)
print(f"\nTotal rows: {len(permits_full)}")
print(f"Date range: {pd.to_datetime(permits_full['issue_date']).min()} to {pd.to_datetime(permits_full['issue_date']).max()}")

2020: pulled 50000 so far...
2020: pulled 51895 so far...
2020: done — 51895 total permits
2021: pulled 50000 so far...
2021: pulled 57421 so far...
2021: done — 57421 total permits


/var/folders/hk/svxlhg9x51j4t3tpg3zjg__00000gn/T/ipykernel_95040/3006262352.py:26: DtypeWarning: Columns (0: ct) have mixed types. Specify dtype option on import or set low_memory=False.
  batch = pd.read_csv(StringIO(response.text))


2022: pulled 50000 so far...
2022: pulled 65336 so far...
2022: done — 65336 total permits
2023: pulled 50000 so far...
2023: pulled 64174 so far...
2023: done — 64174 total permits
2024: pulled 50000 so far...
2024: pulled 64790 so far...
2024: done — 64790 total permits
2025: pulled 50000 so far...
2025: pulled 63435 so far...
2025: done — 63435 total permits

Total rows: 367051
Date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


In [35]:
residential_types = ['Bldg-New', 'Bldg-Addition', 'Bldg-Alter/Repair', 'Bldg-Demolition']
residential_uses = [
    'Dwelling - Single Family',
    'Accessory Dwelling Unit',
    'Apartment',
    'Duplex',
    'Condominium'
]

permits_res = permits_full[
    (permits_full['permit_type'].isin(residential_types)) &
    (permits_full['use_desc'].isin(residential_uses))
].copy()

permits_res['issue_date'] = pd.to_datetime(permits_res['issue_date'])
permits_res['zip_code'] = permits_res['zip_code'].astype(str).str.strip().str.zfill(5)

print(f"Residential permits: {len(permits_res)}")
print(permits_res['permit_type'].value_counts())

Residential permits: 254546
permit_type
Bldg-Alter/Repair    200462
Bldg-Addition         29431
Bldg-New              18860
Bldg-Demolition        5793
Name: count, dtype: int64


In [ ]:
neighborhood_zip_map = {
    'University Park':    ['90007'],
    'Exposition Park':    ['90037'],
    'Jefferson Park':     ['90018'],
    'Pico-Union':         ['90006'],
    'Vermont Square':     ['90044'],
    'Koreatown':          ['90005', '90006', '90010'],
    'West Adams':         ['90016', '90018'],
    'Leimert Park':       ['90008'],
    'Boyle Heights':      ['90033', '90023'],
    'Echo Park':          ['90026'],
    'Silver Lake':        ['90026', '90039'],
    'Los Feliz':          ['90027'],
    'Highland Park':      ['90042'],
    'Glassell Park':      ['90065'],
    'Culver City':        ['90230', '90232'],
    'Mar Vista':          ['90066'],
    'Palms':              ['90034'],
    'West Hollywood':     ['90046', '90069'],
    'Mid-Wilshire':       ['90036'],
    'Hancock Park':       ['90004'],
    'Beverly Hills':      ['90210', '90211'],
    'Brentwood':          ['90049'],
    'Santa Monica':       ['90401', '90403', '90405'],
    'Westwood':           ['90024'],
    'Downtown LA':        ['90012', '90014', '90017'],
}

zip_to_neighborhood = {}
for neighborhood, zips in neighborhood_zip_map.items():
    for z in zips:
        zip_to_neighborhood[z] = neighborhood

permits_res['Neighborhood'] = permits_res['zip_code'].map(zip_to_neighborhood)
permits_mapped = permits_res[permits_res['Neighborhood'].notna()].copy()

print(f"Permits in our neighborhoods: {len(permits_mapped)}")
print(permits_mapped['Neighborhood'].value_counts())

Permits in our neighborhoods: 0
Series([], Name: count, dtype: int64)


In [37]:
# Create year-month period
permits_mapped['Date'] = permits_mapped['issue_date'].dt.to_period('M').dt.to_timestamp('M')

# Clean du_changed — fill NaN with 0
permits_mapped['du_changed'] = pd.to_numeric(permits_mapped['du_changed'], errors='coerce').fillna(0)

# Aggregate
permits_monthly = permits_mapped.groupby(['Neighborhood', 'Date']).agg(
    permits_count=('permit_nbr', 'count'),
    du_added=('du_changed', 'sum')
).reset_index()

print(f"Shape: {permits_monthly.shape}")
print(permits_monthly.head(10))

Shape: (0, 4)
Empty DataFrame
Columns: [Neighborhood, Date, permits_count, du_added]
Index: []


In [38]:
permits_monthly = permits_monthly.sort_values(['Neighborhood', 'Date'])

permits_monthly['permits_trailing6'] = permits_monthly.groupby('Neighborhood')['permits_count'].transform(
    lambda x: x.rolling(6, min_periods=1).sum()
)
permits_monthly['permits_trailing12'] = permits_monthly.groupby('Neighborhood')['permits_count'].transform(
    lambda x: x.rolling(12, min_periods=1).sum()
)
permits_monthly['du_added_trailing6'] = permits_monthly.groupby('Neighborhood')['du_added'].transform(
    lambda x: x.rolling(6, min_periods=1).sum()
)

print(permits_monthly.head(10))

Empty DataFrame
Columns: [Neighborhood, Date, permits_count, du_added, permits_trailing6, permits_trailing12, du_added_trailing6]
Index: []


In [39]:
permits_monthly.to_csv('/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/permits_clean.csv', index=False)
print(f"Saved permits_clean.csv — shape: {permits_monthly.shape}")


Saved permits_clean.csv — shape: (0, 7)


In [41]:
permits_monthly.head()

,Neighborhood,Date,permits_count,du_added,permits_trailing6,permits_trailing12,du_added_trailing6


In [42]:
# Check what zip codes actually look like in the permits data
print("Sample zip codes from permits_res:")
print(permits_res['zip_code'].value_counts().head(20))

# Check what our mapping expects
print("\nSample zip codes from our mapping:")
print(list(zip_to_neighborhood.keys())[:10])

# Check data types
print(f"\nPermits zip_code dtype: {permits_res['zip_code'].dtype}")
print(f"Sample raw values: {permits_res['zip_code'].head(10).tolist()}")

Sample zip codes from permits_res:
zip_code
90026.0    5765
90049.0    5554
91344.0    5425
91335.0    5417
90066.0    5305
90019.0    4835
90042.0    4600
90065.0    4366
90045.0    4312
90016.0    4309
91367.0    4257
90272.0    4186
91406.0    4086
91326.0    3985
90039.0    3933
90034.0    3931
91342.0    3917
91331.0    3707
90043.0    3518
91604.0    3500
Name: count, dtype: int64

Sample zip codes from our mapping:
['90007', '90037', '90018', '90006', '90044', '90005', '90010', '90016', '90008', '90033']

Permits zip_code dtype: str
Sample raw values: ['91307.0', '91342.0', '91411.0', '91607.0', '90003.0', '90025.0', '90034.0', '91324.0', '91214.0', '90064.0']


In [43]:
# Fix ZIP code format - remove decimal and reformat
permits_res['zip_code'] = permits_res['zip_code'].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(5)

# Verify fix
print("Fixed sample zip codes:")
print(permits_res['zip_code'].head(10).tolist())

# Re-map to neighborhoods
permits_res['Neighborhood'] = permits_res['zip_code'].map(zip_to_neighborhood)
permits_mapped = permits_res[permits_res['Neighborhood'].notna()].copy()

print(f"\nPermits in our neighborhoods: {len(permits_mapped)}")
print(permits_mapped['Neighborhood'].value_counts())

Fixed sample zip codes:
['91307', '91342', '91411', '91607', '90003', '90025', '90034', '91324', '91214', '90064']

Permits in our neighborhoods: 74540
Neighborhood
Silver Lake        9698
West Adams         7276
Brentwood          5554
Mar Vista          5305
West Hollywood     4892
Highland Park      4600
Glassell Park      4366
Palms              3931
Los Feliz          3476
Hancock Park       3157
Leimert Park       2881
Vermont Square     2806
Mid-Wilshire       2353
Koreatown          2327
Westwood           2309
Exposition Park    2111
Boyle Heights      1922
Beverly Hills      1850
University Park    1816
Culver City         993
Downtown LA         915
Santa Monica          2
Name: count, dtype: int64


In [45]:
# Aggregate to neighborhood-month
permits_mapped['Date'] = permits_mapped['issue_date'].dt.to_period('M').dt.to_timestamp('M')
permits_mapped['du_changed'] = pd.to_numeric(permits_mapped['du_changed'], errors='coerce').fillna(0)

permits_monthly = permits_mapped.groupby(['Neighborhood', 'Date']).agg(
    permits_count=('permit_nbr', 'count'),
    du_added=('du_changed', 'sum')
).reset_index()

# Compute trailing windows
permits_monthly = permits_monthly.sort_values(['Neighborhood', 'Date'])

permits_monthly['permits_trailing6'] = permits_monthly.groupby('Neighborhood')['permits_count'].transform(
    lambda x: x.rolling(6, min_periods=1).sum()
)
permits_monthly['permits_trailing12'] = permits_monthly.groupby('Neighborhood')['permits_count'].transform(
    lambda x: x.rolling(12, min_periods=1).sum()
)
permits_monthly['du_added_trailing6'] = permits_monthly.groupby('Neighborhood')['du_added'].transform(
    lambda x: x.rolling(6, min_periods=1).sum()
)

# Save
permits_monthly.to_csv('/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/permits_clean.csv', index=False)

print(f"Shape: {permits_monthly.shape}")
print(f"Date range: {permits_monthly['Date'].min()} to {permits_monthly['Date'].max()}")
print(permits_monthly.head(10))

Shape: (1513, 7)
Date range: 2020-01-31 00:00:00 to 2025-12-31 00:00:00
    Neighborhood       Date  permits_count  du_added  permits_trailing6  \
0  Beverly Hills 2020-01-31             41       2.0               41.0   
1  Beverly Hills 2020-02-29             29       2.0               70.0   
2  Beverly Hills 2020-03-31             26       0.0               96.0   
3  Beverly Hills 2020-04-30             12       0.0              108.0   
4  Beverly Hills 2020-05-31             19       0.0              127.0   
5  Beverly Hills 2020-06-30             17      -1.0              144.0   
6  Beverly Hills 2020-07-31             24      -1.0              127.0   
7  Beverly Hills 2020-08-31             23       0.0              121.0   
8  Beverly Hills 2020-09-30             26       1.0              121.0   
9  Beverly Hills 2020-10-31             25       0.0              134.0   

   permits_trailing12  du_added_trailing6  
0                41.0                 2.0  
1             

# DATA COLL 3

In [ ]:
import requests
import pandas as pd

url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

payload = {
    "seriesid": ["SMU06310800000000001"],
    "startyear": "2019",
    "endyear": "2026",
    "registrationkey": BLS_API_KEY
}

response = requests.post(url, json=payload)
data = response.json()

print(f"Status: {data['status']}")
print(f"Message: {data.get('message', 'none')}")

if data['status'] == 'REQUEST_SUCCEEDED':
    series = data['Results']['series'][0]['data']
    employment_df = pd.DataFrame(series)
    print(f"Shape: {employment_df.shape}")
    print(employment_df.head(10))

Status: REQUEST_SUCCEEDED
Message: []
Shape: (88, 6)
   year period periodName latest   value  \
0  2026    M04      April   true  6292.0   
1  2026    M03      March    NaN  6281.6   
2  2026    M02   February    NaN  6258.5   
3  2026    M01    January    NaN  6250.9   
4  2025    M12   December    NaN  6337.7   
5  2025    M11   November    NaN  6338.4   
6  2025    M10    October    NaN  6316.5   
7  2025    M09  September    NaN  6251.9   
8  2025    M08     August    NaN  6250.9   
9  2025    M07       July    NaN  6222.9   

                                footnotes  
0  [{'code': 'P', 'text': 'Preliminary'}]  
1                                    [{}]  
2                                    [{}]  
3                                    [{}]  
4                                    [{}]  
5                                    [{}]  
6                                    [{}]  
7                                    [{}]  
8                                    [{}]  
9                     

In [6]:
employment_df = employment_df.sort_values(['year'])

In [ ]:
# Clean employment data
employment_df['value'] = pd.to_numeric(employment_df['value'], errors='coerce')

# Create proper date column from year and period (M01 = January, M12 = December)
employment_df['Date'] = pd.to_datetime(
    employment_df['year'] + '-' + employment_df['period'].str.replace('M', ''),
    format='%Y-%m'
) + pd.offsets.MonthEnd(0)  # snap to month end to match rent dates

# Sort chronologically
employment_df = employment_df.sort_values('Date').reset_index(drop=True)

# Compute month over month employment growth rate %
employment_df['employment_growth'] = employment_df['value'].pct_change() * 100

# Keep only what we need
employment_clean = employment_df[['Date', 'value', 'employment_growth']].copy()
employment_clean.columns = ['Date', 'employment_level', 'employment_growth']

# Save
employment_clean.to_csv('/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/employment_clean.csv', index=False)

print(f"Shape: {employment_clean.shape}")
print(f"Date range: {employment_clean['Date'].min()} to {employment_clean['Date'].max()}")
print(employment_clean.head(10))

Shape: (88, 3)
Date range: 2019-01-31 00:00:00 to 2026-04-30 00:00:00
        Date  employment_level  employment_growth
0 2019-01-31            6158.2                NaN
1 2019-02-28            6208.7           0.820045
2 2019-03-31            6218.4           0.156232
3 2019-04-30            6214.2          -0.067541
4 2019-05-31            6226.1           0.191497
5 2019-06-30            6233.6           0.120461
6 2019-07-31            6172.9          -0.973755
7 2019-08-31            6199.2           0.426056
8 2019-09-30            6236.7           0.604917
9 2019-10-31            6290.5           0.862636


# DATA COLL 4

In [ ]:
import requests
import pandas as pd
import numpy as np

# Your confirmed 22 neighborhoods and ZIP mapping
neighborhood_zip_map = {
    'University Park':  ['90007'],
    'Exposition Park':  ['90037'],
    'Jefferson Park':   ['90018'],
    'Pico-Union':       ['90006'],
    'Vermont Square':   ['90044'],
    'Koreatown':        ['90005', '90006', '90010'],
    'West Adams':       ['90016', '90018'],
    'Leimert Park':     ['90008'],
    'Boyle Heights':    ['90033', '90023'],
    'Echo Park':        ['90026'],
    'Silver Lake':      ['90026', '90039'],
    'Los Feliz':        ['90027'],
    'Highland Park':    ['90042'],
    'Glassell Park':    ['90065'],
    'Culver City':      ['90230', '90232'],
    'Mar Vista':        ['90066'],
    'Palms':            ['90034'],
    'West Hollywood':   ['90046', '90069'],
    'Mid-Wilshire':     ['90036'],
    'Hancock Park':     ['90004'],
    'Beverly Hills':    ['90210', '90211'],
    'Brentwood':        ['90049'],
    'Santa Monica':     ['90401', '90403', '90405'],
    'Westwood':         ['90024'],
    'Downtown LA':      ['90012', '90014', '90017'],
}

# Filter to your confirmed 22
confirmed_22 = [
    'Beverly Hills', 'Boyle Heights', 'Brentwood', 'Culver City',
    'Downtown LA', 'Exposition Park', 'Glassell Park', 'Hancock Park',
    'Highland Park', 'Koreatown', 'Leimert Park', 'Los Feliz',
    'Mar Vista', 'Mid-Wilshire', 'Palms', 'Santa Monica',
    'Silver Lake', 'University Park', 'Vermont Square', 'West Adams',
    'West Hollywood', 'Westwood'
]

neighborhood_zip_map = {k: v for k, v in neighborhood_zip_map.items() if k in confirmed_22}

# ACS variables we want
# B19013_001E = Median household income
# B25003_002E = Renter occupied units
# B25003_001E = Total occupied units (for renter rate)
# B01003_001E = Total population
# B01001_001E = Total population (for density — we'll use land area separately)

ACS_VARIABLES = {
    'B19013_001E': 'median_income',
    'B25003_001E': 'total_occupied_units',    # total occupied (owner + renter)
    'B25003_002E': 'owner_occupied_units',    # owner occupied
    'B01003_001E': 'population'
}

# ACS 5-year estimates available years
YEARS = [2019, 2020, 2021, 2022, 2023]  # 2024 not yet released

def fetch_acs_zip(year, variables):
    """Fetch ACS 5-year estimates for all ZCTAs."""
    vars_str = ','.join(variables.keys())
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get=NAME,{vars_str}"
        f"&for=zip%20code%20tabulation%20area:*"
        f"&key={CENSUS_API_KEY}"
    )
    
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error {year}: {response.status_code} — {response.text[:200]}")
        return None
    
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns={'zip code tabulation area': 'zip'})
    df = df.rename(columns=variables)
    
    for col in variables.values():
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df['year'] = year
    return df

# Pull all years
all_years = []

for year in YEARS:
    print(f"Fetching {year}...")
    df_year = fetch_acs_zip(year, ACS_VARIABLES)
    if df_year is not None:
        all_years.append(df_year)
        print(f"  {year}: {len(df_year)} ZCTAs fetched")

acs_raw = pd.concat(all_years, ignore_index=True)
print(f"\nTotal rows fetched: {len(acs_raw)}")

# Filter to only ZIPs we need
all_zips = [z for zips in neighborhood_zip_map.values() for z in zips]
acs_filtered = acs_raw[acs_raw['zip'].isin(all_zips)].copy()
print(f"Rows after ZIP filter: {len(acs_filtered)}")

acs_filtered['renter_rate'] = (
    (acs_filtered['total_occupied_units'] - acs_filtered['owner_occupied_units']) 
    / acs_filtered['total_occupied_units']
)

# Map ZIPs to neighborhoods
zip_to_neighborhoods = {}
for neighborhood, zips in neighborhood_zip_map.items():
    for z in zips:
        if z not in zip_to_neighborhoods:
            zip_to_neighborhoods[z] = []
        zip_to_neighborhoods[z].append(neighborhood)

acs_filtered['Neighborhood'] = acs_filtered['zip'].map(
    lambda z: zip_to_neighborhoods.get(z, [None])[0]
)

# Population-weighted aggregation per neighborhood per year
def weighted_avg(group, value_col, weight_col):
    """Compute population-weighted average, handling NaNs."""
    valid = group[[value_col, weight_col]].dropna()
    if valid.empty or valid[weight_col].sum() == 0:
        return np.nan
    return np.average(valid[value_col], weights=valid[weight_col])

records = []

for neighborhood, zips in neighborhood_zip_map.items():
    for year in YEARS:
        subset = acs_filtered[
            (acs_filtered['zip'].isin(zips)) &
            (acs_filtered['year'] == year)
        ]
        
        if subset.empty:
            print(f"  WARNING: No data for {neighborhood} {year}")
            continue
        
        median_income_w = weighted_avg(subset, 'median_income', 'population')
        renter_rate_w   = weighted_avg(subset, 'renter_rate', 'population')
        total_pop       = subset['population'].sum()
        
        records.append({
            'Neighborhood':  neighborhood,
            'year':          year,
            'median_income': round(median_income_w, 2),
            'renter_rate':   round(renter_rate_w, 4),
            'population':    int(total_pop) if not np.isnan(total_pop) else np.nan
        })

demographics = pd.DataFrame(records)

# Sanity checks
print(f"\nShape: {demographics.shape}")
print(f"Neighborhoods: {demographics['Neighborhood'].nunique()}")
print(f"Years: {sorted(demographics['year'].unique())}")
print(f"\nMissing values:\n{demographics.isnull().sum()}")
print(f"\nSample:\n{demographics.head(10)}")

# Save
output_path = '/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/demographics_clean.csv'
demographics.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

Fetching 2019...
  2019: 33120 ZCTAs fetched
Fetching 2020...
  2020: 33120 ZCTAs fetched
Fetching 2021...
  2021: 33774 ZCTAs fetched
Fetching 2022...
  2022: 33774 ZCTAs fetched
Fetching 2023...
  2023: 33772 ZCTAs fetched

Total rows fetched: 167560
Rows after ZIP filter: 170

Shape: (110, 5)
Neighborhoods: 22
Years: [2019, 2020, 2021, 2022, 2023]

Missing values:
Neighborhood     0
year             0
median_income    0
renter_rate      0
population       0
dtype: int64

Sample:
      Neighborhood  year  median_income  renter_rate  population
0  University Park  2019        27406.0       0.8771       42433
1  University Park  2020        30530.0       0.8734       42380
2  University Park  2021        33222.0       0.8815       41270
3  University Park  2022        36326.0       0.8938       41004
4  University Park  2023        36032.0       0.8958       39253
5  Exposition Park  2019        35424.0       0.7591       67640
6  Exposition Park  2020        37945.0       0.7512      

In [12]:
# Check raw values for a few ZIPs
print(acs_filtered[acs_filtered['zip'].isin(['90007', '90005', '90210'])][
    ['zip', 'year', 'total_occupied_units', 'renter_occupied_units']
].head(15))

          zip  year  total_occupied_units  renter_occupied_units
30014   90005  2019                 16781                   1299
30016   90007  2019                 11919                   1465
30079   90210  2019                  7936                   5856
44696   90210  2020                  7752                   6040
46142   90007  2020                 12248                   1550
46172   90005  2020                 16331                   1341
96841   90005  2021                 15993                   1273
96843   90007  2021                 12211                   1447
96906   90210  2021                  7947                   5931
130615  90005  2022                 16607                   1301
130617  90007  2022                 12520                   1329
130680  90210  2022                  7893                   5853
164387  90005  2023                 17477                   1572
164389  90007  2023                 12451                   1297
164452  90210  2023      

# DATA COLL 5